# kline-de-pre-model v3 — Kaggle 一次训练+导出

**v3**: 双通路模型1(CNN+25结构特征→单线性层→3 DCT) + UD zigzag 结构损失 + 锚点平移。

> ⚠️ Settings: **Accelerator 选 GPU**、Internet 打开(clone GitHub)、
> **Add Data 挂上 `mczielinski/bitcoin-historical-data`**(离线读数, 比 OKX 现拉快 25 分钟)。

产出(全部复制到 /kaggle/working): `transformer_dct_v1.pth`、`diffusion_delta_v1.pth`、
`transformer_with_feat_merged.onnx`、`diffusion_unet_merged.onnx`(训练+导出一条龙, 防 12h 截断丢产物)。

In [ ]:
# 1) 拉代码 + 依赖 + GPU 自检
import os
if not os.path.exists('kline-de-pre-model'):
    !git clone --depth 1 https://github.com/lin7c/kline-de-pre-model.git
%cd kline-de-pre-model
!pip -q install 'requests>=2.32' 'pandas>=2.2' 'scipy>=1.13' 'scikit-learn>=1.5'
# Kaggle 当前默认 torch 丢弃了 P100(sm_60) 内核；装一个兼容 P100+T4 的 torch(cu121)
!pip -q install torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121
import torch
print('torch', torch.__version__, '| CUDA avail:', torch.cuda.is_available())
try:
    _ = (torch.zeros(8, device='cuda') + 1).sum().item()   # 真跑一下确认内核可用
    print('✅ GPU 可用:', torch.cuda.get_device_name(0))
except Exception as e:
    os.environ['CUDA_VISIBLE_DEVICES'] = ''                 # 子进程(train.py)会继承 -> 回退 CPU
    print('⚠️ GPU 内核不兼容，本次回退 CPU:', str(e)[:150])

In [ ]:
# 2) 断言测试(窗口语义 + JS/Python 特征一致性) + 构建数据集(离线数据集优先, 无则 OKX 现拉)
!python test_ms_windows.py
!python struct_feat.py
import glob
csvs = sorted(glob.glob('/kaggle/input/**/*1-min*.csv*', recursive=True))
MAX_BARS = '500000'
if csvs:
    print('离线数据:', csvs[0])
    !python getdata_btc.py csv {csvs[0]} {MAX_BARS}
else:
    print('未挂数据集, 从 OKX 现拉(约 25 分钟)')
    !python getdata_btc.py {MAX_BARS} BTC-USDT

In [ ]:
# 3) 训练流水线(限制轮次防 12h 会话截断: 模型1早停很快, 模型2 v8 实测 ~108 轮已平台)
import subprocess, sys, os, time
env = dict(os.environ, CT_EPOCHS='300', UD_EPOCHS='120', UD_STRUCT_LAMBDA='0.1', UD_COND_MIX='0.5')
steps = [
    ('CNN/makedata.py',             '数据预处理(原始窗口)'),
    ('CNN_Transformer/makedata.py', 'DCT标签(v3: 1m×3系数)'),
    ('CNN_Transformer/train.py',    '训练【模型1】双通路CNN'),
    ('CNN_Transformer/gy.py',       '模型1推理 → UD 条件'),
    ('UD/makedata.py',              'UD残差+锚点平移+zigzag枢轴'),
    ('UD/train.py',                 '训练【模型2】UD(zigzag结构损失)'),
]
for script, desc in steps:
    d = os.path.dirname(script) or '.'
    print(f'\n===== {desc} =====', flush=True)
    t0 = time.time()
    subprocess.run([sys.executable, os.path.basename(script)], cwd=d, check=True, env=env)
    print(f'[{desc}] 用时 {(time.time()-t0)/60:.1f} 分钟', flush=True)

In [ ]:
# 4) 导出 ONNX(v3 契约) + 收集全部产物到 /kaggle/working
!pip -q install onnx 2>/dev/null || true
!python export_onnx.py CNN_Transformer/transformer_dct_v1.pth /kaggle/working/transformer_with_feat_merged.onnx
!python export_onnx.py --diffusion UD/diffusion_delta_v1.pth /kaggle/working/diffusion_unet_merged.onnx
import shutil, os
for src in ['CNN_Transformer/transformer_dct_v1.pth', 'UD/diffusion_delta_v1.pth']:
    if os.path.exists(src):
        shutil.copy(src, '/kaggle/working/' + os.path.basename(src))
        print('✅', src, '->', os.path.getsize(src) // 1024, 'KB')
try:
    import onnx
    for f in ['transformer_with_feat_merged.onnx', 'diffusion_unet_merged.onnx']:
        m = onnx.load('/kaggle/working/' + f); onnx.checker.check_model(m)
        print(f, [(i.name, [d.dim_value or d.dim_param for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
except ImportError:
    print('onnx 不可用, 跳过 checker')
!ls -la /kaggle/working/